# QUY TRÌNH TẠO VM + FLOATING IP + SSH

## 0. Load quyền OpenStack

```bash
source ~/tai-openrc.sh
openstack token issue
```

Phải ra token là OK.

---

## 1. Kiểm tra image, flavor, network public

```bash
openstack image list
openstack flavor list
openstack network list
openstack subnet list
```

Cần có:

```text
Image: Ubuntu / Cirros / image bạn dùng
Flavor: tiny / small
Network public: public
Subnet public: 192.168.150.0/24
```

---

## 2. Tạo tenant network cho VM

Ví dụ tạo mạng riêng của bạn tên `tai_network`:

```bash
openstack network create tai_network
```

Kiểm tra:

```bash
openstack network list
```

---

## 3. Tạo subnet cho tenant network

Ví dụ dùng dải `10.10.10.0/24`:

```bash
openstack subnet create tai_subnet \
  --network tai_network \
  --subnet-range 10.10.10.0/24 \
  --gateway 10.10.10.1 \
  --dns-nameserver 8.8.8.8
```

Kiểm tra:

```bash
openstack subnet show tai_subnet
```

Cần thấy:

```text
cidr: 10.10.10.0/24
gateway_ip: 10.10.10.1
```

---

## 4. Tạo router

```bash
openstack router create router1
```

Kiểm tra:

```bash
openstack router list
```

Cần thấy `router1` trạng thái:

```text
ACTIVE / UP
```

---

## 5. Gắn router ra mạng public

```bash
openstack router set router1 --external-gateway public
```

Kiểm tra:

```bash
openstack router show router1
```

Cần thấy:

```text
external_gateway_info: network_id public
```

---

## 6. Gắn subnet VM vào router

```bash
openstack router add subnet router1 tai_subnet
```

Kiểm tra:

```bash
openstack router show router1
```

Cần thấy trong `interfaces_info` có:

```text
10.10.10.1
```

Đây chính là gateway của VM.

---

## 7. Tạo keypair để SSH

```bash
openstack keypair create tai_key > ~/tai_key.pem
chmod 400 ~/tai_key.pem
```

Kiểm tra:

```bash
openstack keypair list
```

Cần thấy:

```text
tai_key
```

---

## 8. Tạo security group

```bash
openstack security group create tai_sg \
  --description "Allow SSH and ICMP"
```

Mở ping:

```bash
openstack security group rule create \
  --protocol icmp \
  --remote-ip 0.0.0.0/0 \
  tai_sg
```

Mở SSH:

```bash
openstack security group rule create \
  --protocol tcp \
  --dst-port 22 \
  --remote-ip 0.0.0.0/0 \
  tai_sg
```

Kiểm tra:

```bash
openstack security group rule list tai_sg
```

Cần thấy:

```text
icmp ingress 0.0.0.0/0
tcp 22 ingress 0.0.0.0/0
egress 0.0.0.0/0
```

---

## 9. Tạo VM

Ví dụ:

```bash
openstack server create \
  --flavor small \
  --image "Ubuntu 22.04 LTS" \
  --network tai_network \
  --key-name tai_key \
  --security-group tai_sg \
  vm-tai-01
```

Theo dõi trạng thái:

```bash
watch -n 2 "openstack server show vm-tai-01 -c status -c addresses"
```

Đợi đến khi:

```text
status: ACTIVE
addresses: tai_network=10.10.10.x
```

Thoát `watch` bằng `Ctrl + C`.

---

## 10. Kiểm tra console log VM

```bash
openstack console log show vm-tai-01 | tail -50
```

Cần thấy cloud-init chạy xong hoặc không có lỗi nghiêm trọng.

---

## 11. Tạo Floating IP

```bash
openstack floating ip create public
```

Ví dụ nhận được:

```text
192.168.150.16
```

Kiểm tra:

```bash
openstack floating ip list
```

---

## 12. Gán Floating IP vào VM

```bash
openstack server add floating ip vm-tai-01 192.168.150.16
```

Kiểm tra:

```bash
openstack floating ip list
openstack server show vm-tai-01 -c addresses
```

Cần thấy:

```text
10.10.10.x
192.168.150.16
```

---

## 13. Kiểm tra trong VM qua console

Vào console nếu cần:

```bash
openstack console url show vm-tai-01
```

Hoặc dùng Horizon console.

Trong VM kiểm tra:

```bash
ip a
ip route
```

Cần thấy:

```text
10.10.10.x/24
default via 10.10.10.1
```

Test gateway:

```bash
ping -c 3 10.10.10.1
```

Nếu ping được gateway là VM ↔ router OK.

---

## 14. Test Floating IP từ controller hoặc máy ngoài

```bash
ping -c 3 192.168.150.16
```

Nếu ping được, SSH:

```bash
ssh -i ~/tai_key.pem ubuntu@192.168.150.16
```

Với Cirros thì user thường là:

```bash
ssh -i ~/tai_key.pem cirros@192.168.150.16
```

---

# CHECKLIST KHI KHÔNG SSH ĐƯỢC

## 1. VM có ACTIVE không?

```bash
openstack server show vm-tai-01 -c status
```

Phải là:

```text
ACTIVE
```

---

## 2. VM có IP nội bộ không?

```bash
openstack server show vm-tai-01 -c addresses
```

Phải có:

```text
10.10.10.x
```

---

## 3. Router có nối subnet không?

```bash
openstack router show router1
```

Phải có:

```text
10.10.10.1
external_gateway_info: public
```

---

## 4. Security group đã mở chưa?

```bash
openstack security group rule list tai_sg
```

Phải có:

```text
ICMP
TCP 22
```

---

## 5. Floating IP đã map đúng chưa?

```bash
openstack floating ip list
```

Phải có:

```text
192.168.150.x  ->  10.10.10.x
```

---

## 6. VM nằm trên compute nào?

```bash
openstack port show <PORT_ID>
```

Xem dòng:

```text
binding_host_id
```

Nếu VM nằm trên `stack3`, thì Floating IP path chạy qua `stack3`.

---

## 7. br-ex trên compute đó có uplink chưa?

Trên node compute chứa VM:

```bash
sudo ovs-vsctl show
```

`br-ex` phải có uplink ra mạng public, ví dụ:

```text
Bridge br-ex
    Port eno8303
    Port br-ex
```

Nếu `br-ex` chỉ có `dummy0` thì Floating IP sẽ không hoạt động.

---

# TÓM TẮT LUỒNG ĐÚNG

```text
1. Tạo tenant network
2. Tạo tenant subnet
3. Tạo router
4. Router set external gateway public
5. Router add subnet tenant
6. Tạo keypair
7. Tạo security group mở ICMP + SSH
8. Tạo VM
9. Đợi VM ACTIVE
10. Tạo Floating IP
11. Gán Floating IP vào VM
12. Ping Floating IP
13. SSH vào VM
```

Câu quan trọng cần nhớ:

```text
Floating IP không nằm trong VM.
Floating IP là NAT ở Neutron router/OVN.
VM chỉ thấy IP private, ví dụ 10.10.10.24.
```
